In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — Setup: Schemas and Tables
# MAGIC **Run once before anything else. Safe to re-run (all statements use IF NOT EXISTS).**
# MAGIC
# MAGIC Creates the Bronze, Silver, and Gold schemas and their tables in the
# MAGIC `bootcamp_students` catalog.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Create Schemas

# COMMAND ----------

CATALOG = "bootcamp_students"

for layer in ["bronze", "silver", "gold"]:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {CATALOG}.{layer}
        COMMENT 'ARC LAB {layer.capitalize()} layer — WattTime carbon intensity pipeline'
    """)
    print(f"✓ {CATALOG}.{layer}")

print("\nAll schemas ready.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Create Bronze Table
# MAGIC Raw WattTime API responses. Append-only. Never modified after write.

# COMMAND ----------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.watttime_raw (
        region_code     STRING        COMMENT 'WattTime balancing area e.g. CAISO_NORTH',
        signal_type     STRING        COMMENT 'Signal type e.g. co2_moer',
        ts_utc          TIMESTAMP_NTZ COMMENT 'Signal timestamp in UTC',
        value           DOUBLE        COMMENT 'Raw signal value from API',
        units           STRING        COMMENT 'Units e.g. lbs_co2_per_mwh',
        model_version   STRING        COMMENT 'WattTime model version e.g. 2023-03-01',
        api_endpoint    STRING        COMMENT 'Endpoint called e.g. /v3/historical',
        ingested_at     TIMESTAMP_NTZ COMMENT 'When this record was written to Bronze',
        ingestion_date  DATE          COMMENT 'Partition column — derived from ingested_at',
        source_file     STRING        COMMENT 'Tag identifying ingestion run e.g. backfill_7day'
    )
    USING DELTA
    PARTITIONED BY (ingestion_date)
    COMMENT 'ARC LAB Bronze — raw WattTime API responses, append-only'
""")
print(f"✓ {CATALOG}.bronze.watttime_raw")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Create Silver Table
# MAGIC Cleaned, deduplicated, quality-flagged data. One record per 5-min bucket.

# COMMAND ----------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.silver.watttime_clean (
        region_code           STRING        COMMENT 'WattTime balancing area',
        signal_type           STRING        COMMENT 'Signal type',
        ts_utc                TIMESTAMP_NTZ COMMENT 'Signal timestamp UTC',
        ts_local              TIMESTAMP_NTZ COMMENT 'Signal timestamp America/Los_Angeles',
        ts_date               DATE          COMMENT 'Partition column — date of signal',
        ts_5min_bucket        TIMESTAMP_NTZ COMMENT 'Timestamp rounded down to 5-min boundary',
        lbs_co2_per_mwh       DOUBLE        COMMENT 'Carbon intensity in lbs CO2 per MWh',
        model_version         STRING        COMMENT 'WattTime model version',
        model_transition_flag BOOLEAN       COMMENT 'True if record is on or after March 18 2026 model update',
        range_flag            BOOLEAN       COMMENT 'True if value is outside expected bounds',
        quality_flag          STRING        COMMENT 'PASS or WARN',
        ingested_at           TIMESTAMP_NTZ COMMENT 'Inherited from Bronze',
        processed_at          TIMESTAMP_NTZ COMMENT 'When Silver processing ran'
    )
    USING DELTA
    PARTITIONED BY (ts_date)
    COMMENT 'ARC LAB Silver — cleaned, deduplicated, quality-flagged WattTime data'
""")
print(f"✓ {CATALOG}.silver.watttime_clean")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Create Gold Table
# MAGIC Hourly aggregates with scheduling opportunity metrics.

# COMMAND ----------

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.gold.carbon_intensity_hourly (
        region_code                  STRING    COMMENT 'WattTime balancing area',
        hour_of_day                  INT       COMMENT 'Hour 0-23 UTC',
        is_weekend                   BOOLEAN   COMMENT 'True = Saturday or Sunday',
        avg_intensity                DOUBLE    COMMENT 'Average lbs CO2/MWh for this hour slot',
        min_intensity                DOUBLE    COMMENT 'Historical minimum',
        max_intensity                DOUBLE    COMMENT 'Historical maximum',
        stddev_intensity             DOUBLE    COMMENT 'Standard deviation — variability measure',
        p25_intensity                DOUBLE    COMMENT '25th percentile intensity',
        p50_intensity                DOUBLE    COMMENT 'Median intensity',
        p75_intensity                DOUBLE    COMMENT '75th percentile intensity',
        sample_count                 LONG      COMMENT 'Number of raw 5-min readings in this bucket',
        threshold_p20                DOUBLE    COMMENT 'Region 20th percentile — low-carbon cutoff',
        threshold_p80                DOUBLE    COMMENT 'Region 80th percentile — high-carbon cutoff',
        region_min_avg               DOUBLE    COMMENT 'Best hourly average across all hours',
        region_max_avg               DOUBLE    COMMENT 'Worst hourly average across all hours',
        is_low_carbon_window         BOOLEAN   COMMENT 'True = best hours to run AI workloads',
        is_high_carbon_window        BOOLEAN   COMMENT 'True = worst hours to run AI workloads',
        avoidable_lbs_co2_per_mwh    DOUBLE    COMMENT 'Carbon saved per MWh by shifting to optimal window',
        scheduling_opportunity_score DOUBLE    COMMENT '0-100 score: higher = more benefit from intelligent scheduling',
        pipeline_version             STRING    COMMENT 'Pipeline version for lineage tracking',
        updated_at                   TIMESTAMP COMMENT 'Last pipeline run timestamp'
    )
    USING DELTA
    COMMENT 'ARC LAB Gold — hourly carbon intensity aggregates and AI scheduling opportunity metrics'
""")
print(f"✓ {CATALOG}.gold.carbon_intensity_hourly")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Verify

# COMMAND ----------

print("=== SCHEMA VERIFICATION ===\n")
for layer in ["bronze", "silver", "gold"]:
    tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{layer}").collect()
    for t in tables:
        print(f"✓ {CATALOG}.{layer}.{t.tableName}")

print("\nSetup complete. Ready to ingest.")

✓ bootcamp_students.bronze
✓ bootcamp_students.silver
✓ bootcamp_students.gold

All schemas ready.
✓ bootcamp_students.bronze.watttime_raw
✓ bootcamp_students.silver.watttime_clean
✓ bootcamp_students.gold.carbon_intensity_hourly
=== SCHEMA VERIFICATION ===

✓ bootcamp_students.bronze.watttime_raw
✓ bootcamp_students.silver.watttime_clean
✓ bootcamp_students.gold.carbon_intensity_hourly

Setup complete. Ready to ingest.
